# SANKET — INCLUDE landmark extraction on Google Colab

Ghar pe Zenodo se download ~67 kB/s (≈10 din). Colab Google ke data centre mein hai, isliye download bahut tez hota hai.

**Ye notebook kya karta hai:** 44 zips ko **ek-ek karke** download karta hai → unzip → har video par MediaPipe Holistic (same code jo app mein hai) → har video ke landmarks (~100 KB `.npz`) seedhe **Google Drive** mein save → zip + videos delete. Agla zip background mein pehle se download hota rehta hai. Disk par kabhi ~3 GB se zyada nahi.

**Runtime:** `Runtime → Change runtime type → CPU` (GPU ki zaroorat nahi — MediaPipe CPU par chalta hai, aur TARKA ke compute units bache rehte hain). High-RAM mile to behtar (zyada cores).

**Session toot gaya?** Bas upar se saare cells dobara chalao — jo zips ho chuke hain wo Drive ke markers ki wajah se skip ho jaate hain.

**Andaaza:** download ~30 min – 2 ghante (Zenodo ki speed par) + extraction ~1.5–3 ghante.

In [ ]:
# 1) Google Drive mount — landmarks yahan save honge taaki session tootne par bhi kuch na khoye.
import pathlib

from google.colab import drive

drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/SANKET')
CACHE = DRIVE / 'cache'
CACHE.mkdir(parents=True, exist_ok=True)
print('cache folder:', CACHE, '| already cached:', sum(1 for _ in CACHE.rglob('*.npz')), 'files')

In [ ]:
# 2) SANKET ka code laao (public repo) + sirf zaroori packages. Colab mein numpy/opencv pehle se hain.
import multiprocessing
import shutil

!if [ -d /content/SANKET ]; then git -C /content/SANKET pull -q; else git clone -q https://github.com/Kanak234/SANKET /content/SANKET; fi
!pip install -q mediapipe requests tqdm
!cd /content/SANKET && python scripts/fetch_models.py --mediapipe-only
print('CPU cores:', multiprocessing.cpu_count(), '| free disk GB:', round(shutil.disk_usage('/content').free / 1e9, 1))

In [ ]:
# 3) MAIN STEP — zip-by-zip download + landmark extraction (resumable). Isme ghante lagenge; tab khula rakho.
#    Pehle sirf ek chhota test chahiye to neeche ONLY = 'Greetings' kar do.
ONLY = ''          # '' = saare 44 zips
WORKERS = multiprocessing.cpu_count()
only_arg = f'--only {ONLY}' if ONLY else ''
!cd /content/SANKET && SANKET_CACHE="{CACHE}" PYTHONPATH=/content/SANKET python -m training.colab_stream --workers {WORKERS} {only_arg}

In [ ]:
# 4) Poora cache ek zip mein pack karo (Drive par SANKET_cache.zip, ~400 MB) — ghar par yahi download karna hai.
import pathlib
import shutil

n = sum(1 for _ in CACHE.rglob('*.npz'))
print('landmark files:', n, '(expected ~4,290)')
fail = CACHE / '_extract_failures.csv'
if fail.exists():
    print('failed videos:', sum(1 for _ in fail.open()), '→', fail)
out = shutil.make_archive(str(DRIVE / 'SANKET_cache'), 'zip', root_dir=str(DRIVE), base_dir='cache')
print('saved', out, round(pathlib.Path(out).stat().st_size / 1e6), 'MB')

## Ghar par (i9 + RTX 5060 Ti)

1. Google Drive → `MyDrive/SANKET/SANKET_cache.zip` download karo (Downloads folder mein).
2. PowerShell mein ek command:

```powershell
Set-Location D:\Projects\sanket; git pull; powershell -ExecutionPolicy Bypass -File .\scripts\run_full_pipeline.ps1 -FromCache
```

Ye cache unzip karke **download/extraction skip** karta hai aur seedha training → evaluation → ONNX → avatar → self-test chalata hai (~1 ghanta).